In [ ]:
install.packages("ggm")

In [ ]:
if (!requireNamespace("BiocManager", quietly = TRUE)) {
    install.packages("BiocManager")
}

# Install ggm and its dependencies using BiocManager, which handles Bioconductor dependencies like 'graph'
BiocManager::install("ggm", dependencies = TRUE)

library(ggm)

packageVersion("ggm")

In [ ]:
help(package="ggm")

In [ ]:
library(ggm)

g1 <- DAG(
  y ~ x + z,
  x ~ z
)

g1

In [ ]:
class(g1)
str(g1)

In [ ]:
pa(g2, "y")

In [ ]:
g3 <- DAG(
  y ~ x,
  x ~ z
)

In [ ]:
dSep(
  g3,
  X = "z",
  Y = "y",
  Z = c("x")
)

In [ ]:
dSep(g3,"z","y",NULL)

In [ ]:
######## Colliders ########
g4 <- DAG(
  c ~ a + b
)

In [ ]:
dSep(g4,"a","b",NULL)

In [ ]:
dSep(g4,"a","b","c")

In [ ]:
methods(class=class(g5))

In [ ]:
MarkEqMag
MarkEqRcg

In [ ]:
?MarkEqMag
?makeMG
?AG

In [ ]:
############################################################
# GGM exploration script
# Install once, then load
############################################################

# install.packages("ggm")   # uncomment if needed
library(ggm)

############################################################
# 0. Quick package inspection
############################################################

packageVersion("ggm")
help(package = "ggm")

# See function signatures in your installed version
args(DAG)
args(UG)
args(makeMG)

args(pa)
args(dSep)
args(msep)

args(AG)
args(SG)
args(RG)

args(MarkEqMag)

############################################################
# Helper: print graph nicely
############################################################

show_graph <- function(g, name = "graph") {
  cat("\n---", name, "---\n")
  print(g)
  cat("\nclass: ")
  print(class(g))
  cat("\ndim: ")
  print(dim(g))
  cat("\nrownames:\n")
  print(rownames(g))
  cat("\ncolnames:\n")
  print(colnames(g))
  cat("\n")
}

############################################################
# 1. DAG()
############################################################
# The handbook defines DAGs from model formulae, e.g.
# DAG(Y ~ X + U, X ~ Z + U)
# and says the result is an adjacency matrix with labels. [handbook]

dag1 <- DAG(Y ~ X + U, X ~ Z + U)
show_graph(dag1, "dag1 from DAG()")

# Compare with edge matrix representation
edgeMatrix(dag1)

# If dimnames are missing in your version, force them manually:
if (is.null(dimnames(dag1))) {
  vars <- c("Y", "X", "U", "Z")
  dimnames(dag1) <- list(vars, vars)
}

show_graph(dag1, "dag1 after forcing dimnames")

# Parent query
pa(dag1, "Y")
pa(dag1, "X")

# d-separation tests
# Handbook syntax: dSep(dag, first="Z", second="U", cond="Y")
dSep(dag1, first = "Z", second = "U", cond = "Y")
dSep(dag1, first = "Z", second = "U", cond = NULL)

############################################################
# 2. UG()
############################################################
# Start by checking the signature in your installation:
args(UG)
?UG

# If UG accepts formula syntax in your version, try:
# ug1 <- UG(X ~ Y + Z, Y ~ Z)
# show_graph(ug1, "ug1 from UG()")

# Fallback: build an undirected graph manually as an adjacency matrix
vars2 <- c("A", "B", "C", "D")
ug2 <- matrix(0, nrow = 4, ncol = 4, dimnames = list(vars2, vars2))

# Add undirected edges by setting symmetric entries to 1
ug2["A", "B"] <- 1; ug2["B", "A"] <- 1
ug2["B", "C"] <- 1; ug2["C", "B"] <- 1
ug2["C", "D"] <- 1; ug2["D", "C"] <- 1

show_graph(ug2, "ug2 manual undirected graph")

############################################################
# 3. makeMG()
############################################################
# Mixed graphs may contain directed, undirected, and bidirected edges.
# First inspect the constructor in your version:
args(makeMG)
?makeMG

# Try a small mixed graph manually if the constructor is unclear.
# In ggm, adjacency codes in the handbook are:
# 0 = no edge
# 1 = undirected
# 2 = bidirected
# 3 = part of a double edge / bow pattern [handbook]

vars3 <- c("X", "Y", "Z", "W")
mg <- matrix(0, nrow = 4, ncol = 4, dimnames = list(vars3, vars3))

# Example mixed graph:
# X -> Y
# Y -- Z
# Z <-> W
mg["Y", "X"] <- 1        # X -> Y  means e_{Y,X} = 1 and e_{X,Y} = 0
mg["Y", "Z"] <- 1; mg["Z", "Y"] <- 1   # undirected Y -- Z
mg["Z", "W"] <- 2; mg["W", "Z"] <- 2   # bidirected Z <-> W

show_graph(mg, "manual mixed graph")

############################################################
# 4. pa()
############################################################
# pa() returns parents of a node in a DAG.
# The handbook uses named adjacency matrices, so dimnames matter. [handbook]

g2 <- DAG(Y ~ X + U, X ~ Z + U)

# Make sure names exist and match case exactly
if (is.null(dimnames(g2))) {
  vars <- c("Y", "X", "U", "Z")
  dimnames(g2) <- list(vars, vars)
}

rownames(g2)
colnames(g2)

# Use the exact vertex name as stored
pa(g2, "Y")
pa(g2, "X")
pa(g2, "U")
pa(g2, "Z")

# If your version still errors, inspect:
str(g2)
class(g2)
dimnames(g2)

############################################################
# 5. dSep()
############################################################
# Handbook syntax:
# dSep(dag, first="Z", second="U", cond="Y")
# not X/Y/Z. [handbook]

g3 <- DAG(Y ~ X, X ~ Z)

if (is.null(dimnames(g3))) {
  vars <- c("Y", "X", "Z")
  dimnames(g3) <- list(vars, vars)
}

show_graph(g3, "g3")

dSep(g3, first = "Z", second = "Y", cond = "X")   # should be TRUE for Z -> X -> Y
dSep(g3, first = "Z", second = "Y", cond = NULL)   # should be FALSE
dSep(g3, first = "Z", second = "X", cond = NULL)   # adjacent, so FALSE

# Collider example
g4 <- DAG(C ~ A + B)
if (is.null(dimnames(g4))) {
  vars <- c("A", "B", "C")
  dimnames(g4) <- list(vars, vars)
}
show_graph(g4, "collider g4")

dSep(g4, first = "A", second = "B", cond = NULL)   # TRUE
dSep(g4, first = "A", second = "B", cond = "C")    # FALSE

############################################################
# 6. msep()
############################################################
# msep() is for separation in mixed graphs / ancestral-type settings
# in your installed version. First inspect its signature:
args(msep)
?msep

# Example usage will depend on the exact argument names in your version.
# Use msep() only after checking args(msep).
# For now, print the help and try one example from the manual/package docs.
############################################################

############################################################
# 7. AG(), SG(), RG()
############################################################
# These are graph constructors / graph classes in the package ecosystem.
# Check signatures first, because syntax can vary by version.

args(AG)
args(SG)
args(RG)

?AG
?SG
?RG

# If they accept adjacency matrices, you can try:
# ag <- AG(mg)
# sg <- SG(mg)
# rg <- RG(mg)
#
# If they accept formulae or named arguments, use the help page examples.
############################################################

############################################################
# 8. MarkEqMag()
############################################################
# Markov equivalent maximal ancestral graphs / MAGs.
# Inspect signature and examples first.

args(MarkEqMag)
?MarkEqMag

# Typical workflow:
# 1) create a graph
# 2) call MarkEqMag(graph) or as documented in your version
############################################################

############################################################
# 9. Useful debugging patterns
############################################################

# When a function fails, inspect the graph:
str(dag1)
attributes(dag1)

# Check exact names (case-sensitive):
rownames(dag1)
colnames(dag1)

# If a function expects dimnames, ensure them:
stopifnot(!is.null(dimnames(dag1)))

############################################################
# 10. Small summary checks
############################################################

# DAG check
edgeMatrix(dag1)

# More inspection
summary(dag1)